In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

**⚠️ Data Requirement — BioImageArchive:** This notebook requires raw microscopy data from BioImageArchive. Download the dataset and set `data_archive_path` to your local BioImageArchive directory (see README).

**Pipeline step 2/5 (replicate 1)** — run `0_combined_df_wt.ipynb` first.

**Run analysis scripts in this order:**
1. `0_combined_df_wt.ipynb` — load and combine raw image data
2. `1_fluo_binary_wt_rep1/2/3.ipynb` — extract fluorescence binary data (run for each replicate)
3. `1b_generate_masks_rep1/2/3.ipynb` — generate segmentation masks (run for each replicate)
4. `2_first_colonies_wt_rep1/2/3.ipynb` — annotate first-appearance colonies (run for each replicate)
5. `3_calculate_regrowth_fractions.ipynb` — calculate regrowth fractions

# Fluorescence Normalization for Wild-Type Replicate 1

## Overview
This notebook processes fluorescence intensity data for replicate 1 of the wild-type strain, performing background subtraction and normalization.

## Workflow
1. **Data Loading**: Loads fluorescence data for `replicate_1` and background measurements
2. **Background Calculation**: Computes average background intensity from frames 0–8
3. **Normalization**: Normalizes fluorescence signal per cell using:
   - Background subtraction
   - Normalization to frame 0 mean intensity
4. **Output**: Stores normalized values in a new `fluo_norm` column

In [ ]:
# Define file paths (relative to notebook location) and parameters
file_path = '0_combined_df_wt.csv'
file_path_background = '../../../Figure2/background/0_combined_df_bg.csv'
replicate = 'replicate_1'

# Function definitions
def load_data(file_path, file_path_background):
    """Load experimental and background data for the specified replicate."""
    df = pd.read_csv(file_path)
    df = df[df['replicate'] == replicate]
    df_bg = pd.read_csv(file_path_background)
    return df, df_bg

def calculate_bg_intensity(df_bg, frame_range):
    """Calculate mean background intensity over specified frame range."""
    return df_bg[df_bg['frame_number'].isin(frame_range)]['intensity_raw_mcherry'].mean()

def calculate_mean_intensity(df, frame_number):
    """Calculate mean intensity at a specific frame number."""
    return df[df['frame_number'] == frame_number]['intensity_raw_mcherry'].mean()

def normalize_fluorescence(df, bg_intensity, mean_intensity_frame_1):
    """Normalize fluorescence by background subtraction and frame 0 normalization."""
    df['fluo_norm'] = (df['intensity_raw_mcherry'] - bg_intensity) / (mean_intensity_frame_1 - bg_intensity)

# Execute normalization pipeline
df, df_bg = load_data(file_path, file_path_background)
bg_intensity = calculate_bg_intensity(df_bg, range(0, 9))
mean_intensity_frame_1 = calculate_mean_intensity(df, 0)
normalize_fluorescence(df, bg_intensity, mean_intensity_frame_1)

print(f"Background intensity: {bg_intensity:.2f}")
print(f"Frame 0 mean intensity: {mean_intensity_frame_1:.2f}")
print(f"Normalized {len(df)} measurements")

# Threshold Determination Using KDE-Based Valley Detection

## Method
This analysis computes an optimal threshold for classifying cells as fluorescent or non-fluorescent using kernel density estimation (KDE).

## Algorithm
1. **Data Selection**: Extract normalized fluorescence values at frame 100
2. **KDE Analysis**:
   - Compute KDE curve over the intensity distribution
   - Identify valleys (local minima) in the KDE
   - Locate the highest peak to the left of the first valley
3. **Threshold Calculation**: Set threshold as the midpoint between the left peak and valley
4. **Visualization**: Display histogram, KDE curve, and threshold line

## Rationale
This method effectively separates low and high intensity populations in bimodal distributions, providing an objective threshold for binary classification.

In [ ]:
# Define the frame used for thresholding and visualization
threshold_frame = 100

# Initialize figure
plt.figure(figsize=(10, 5))

# Filter data for the specified frame
frame = df[df['frame_number'] == threshold_frame]
fluo_norm_values = frame['fluo_norm'].dropna().values

threshold = np.nan  # Default if no threshold found

if len(fluo_norm_values) > 0:
    # Compute kernel density estimate
    kde = gaussian_kde(fluo_norm_values)
    x_range = np.linspace(min(fluo_norm_values), max(fluo_norm_values), 1000)
    kde_values = kde(x_range)

    # Find valleys (minima) in the KDE
    valleys, _ = find_peaks(-kde_values)

    if len(valleys) > 0:
        valley_idx = valleys[0]
        valley_x = x_range[valley_idx]

        # Find peaks (local maxima) in the KDE
        peaks, _ = find_peaks(kde_values)

        # Identify the highest peak to the left of the valley
        left_peaks = [p for p in peaks if p < valley_idx]
        if left_peaks:
            left_peak_idx = max(left_peaks, key=lambda i: kde_values[i])
            left_peak_x = x_range[left_peak_idx]
            threshold = (left_peak_x + valley_x) / 2
        else:
            # Fallback if no left peak found
            threshold = valley_x

    # Visualize the distribution and threshold
    plt.hist(fluo_norm_values, bins=50, alpha=0.3, density=True, label='Histogram')
    plt.plot(x_range, kde_values, linewidth=2, label='KDE')
    plt.axvline(threshold, color='r', linestyle='dashed', linewidth=2, label=f'Threshold = {threshold:.3f}')

plt.title(f'Fluorescence Intensity Distribution at Frame {threshold_frame}')
plt.xlabel('Normalized Fluorescence Intensity')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Computed threshold value: {threshold:.4f}")

# Binary Classification and Data Export

## Classification
Applies the KDE-derived threshold to classify all cells in the dataset:
- `'r'` (red/fluorescent): `fluo_norm > threshold`
- `'b'` (black/non-fluorescent): `fluo_norm ≤ threshold`

## Output
Saves the enhanced dataframe with the `fluo_binary` classification column to CSV.

In [ ]:
# Classification function
def classify_fluorescence(df, threshold):
    """Classify cells as fluorescent ('r') or non-fluorescent ('b') based on threshold."""
    df['fluo_binary'] = np.where(df['fluo_norm'] > threshold, 'r', 'b')
    return df

# Apply classification using the KDE-derived threshold
df = classify_fluorescence(df, threshold)

# Save results to CSV (relative path)
output_path = '1_fluo_binary_wt_rep1.csv'
df.to_csv(output_path, index=False)

# Display summary statistics
n_fluorescent = (df['fluo_binary'] == 'r').sum()
n_non_fluorescent = (df['fluo_binary'] == 'b').sum()
print(f"Classification complete:")
print(f"  Fluorescent cells: {n_fluorescent} ({100*n_fluorescent/len(df):.1f}%)")
print(f"  Non-fluorescent cells: {n_non_fluorescent} ({100*n_non_fluorescent/len(df):.1f}%)")
print(f"Output saved to: {output_path}")